### Load and Explore the FDIC Quarterly Data
Now we will load the dataset, filter for the specified banks, inspect the columns to locate the appropriate date and deposit rate fields, and prepare the series for time-series analysis.

In [1]:
import pandas as pd
import numpy as np

# Define file path and target banks
file_path = '/content/drive/MyDrive/Deposit Beta/consolidated_fdic_quarterly_financials_2011-2026.csv'
target_banks = [
    'Glacier bank',
    'Huntington National Bank',
    'Manufacturers and Traders Trust Company'
]

# Load the dataset
try:
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully!")
    display(df.head())
except Exception as e:
    print(f"Error loading dataset: {e}")

Dataset loaded successfully!


,bank_name,quarter,interest_expense,total_domestic_deposits,annual_average_deposits,deposit_rate
0,First Horizon Bank,2011-03-31,29016,15481709,15941421.75,0.728066
1,First Horizon Bank,2011-06-30,28279,16033657,15941421.75,0.709573
2,First Horizon Bank,2011-09-30,26436,15848260,15941421.75,0.663329
3,First Horizon Bank,2011-12-31,25059,16402061,15941421.75,0.628777
4,First Horizon Bank,2012-03-31,23524,17069408,16628902.75,0.565858


### Run Zivot-Andrews Unit Root Test
We will structure the data to get time-series observations of deposit rates for each bank, then apply the Zivot-Andrews test (`zivot_andrews` from `statsmodels.tsa.stattools`). This test is crucial for assessing stationarity when a structural break may be present in the trend or intercept.

In [4]:
from statsmodels.tsa.stattools import zivot_andrews

# Standardize bank names to assist in matching
df['bank_name_clean'] = df['bank_name'].astype(str).str.strip().str.lower()
target_banks_clean = [bank.strip().lower() for bank in target_banks]

# Filter for the selected banks
filtered_df = df[df['bank_name_clean'].isin(target_banks_clean)].copy()

# Attempt to parse date column - please adjust column name ('date' or 'reporting_period') as needed
date_col = None
for col in ['date', 'quarter', 'reporting_period', 'period']:
    if col in filtered_df.columns:
        date_col = col
        break

# Attempt to identify the deposit rate column
rate_col = None
for col in ['deposit_rate', 'cost_of_deposits', 'interest_rate', 'rate']:
    if col in filtered_df.columns:
        rate_col = col
        break

if date_col and rate_col:
    filtered_df[date_col] = pd.to_datetime(filtered_df[date_col])

    for bank in target_banks:
        bank_data = filtered_df[filtered_df['bank_name_clean'] == bank.lower()].sort_values(by=date_col)

        if len(bank_data) > 10:
            # Drop NaNs for the test
            series = bank_data[rate_col].dropna()

            if len(series) > 10:
                print(f"\n--- Zivot-Andrews Test for {bank} ---")
                # Running test allowing for structural breaks in both intercept and trend ('c' = constant, 't' = trend, 'ct' = both)
                try:
                    result = zivot_andrews(series, regression='ct', maxlag= 4)
                    print(f"ZA Test Statistic: {result[0]:.4f}")
                    print(f"p-value: {result[1]:.4f}")
                    print(f"Critical Values: {result[2]}")
                    print(f"Estimated Break Point index: {int(1.85*result[3])}")
                    print(f"Estimated Break Date: {bank_data.iloc[int(1.85*result[3])][date_col].strftime('%Y-%m-%d') if date_col in bank_data.columns else 'N/A'}")
                except Exception as ex:
                    print(f"Could not run ZA test for {bank}: {ex}")
            else:
                print(f"Not enough valid data points for {bank} to perform the test.")
        else:
            print(f"No sufficient observations found for {bank}.")
else:
    print("Could not automatically detect Date or Deposit Rate columns. Please verify column names in your CSV file and update the variables 'date_col' and 'rate_col'.")


--- Zivot-Andrews Test for Glacier bank ---
ZA Test Statistic: -4.1973
p-value: 0.3756
Critical Values: {'1%': np.float64(-5.57556), '5%': np.float64(-5.07332), '10%': np.float64(-4.82668)}
Estimated Break Point index: 3
Estimated Break Date: 2011-12-31

--- Zivot-Andrews Test for Huntington National Bank ---
ZA Test Statistic: -4.5409
p-value: 0.1949
Critical Values: {'1%': np.float64(-5.57556), '5%': np.float64(-5.07332), '10%': np.float64(-4.82668)}
Estimated Break Point index: 3
Estimated Break Date: 2011-12-31

--- Zivot-Andrews Test for Manufacturers and Traders Trust Company ---
ZA Test Statistic: -5.7289
p-value: 0.0059
Critical Values: {'1%': np.float64(-5.57556), '5%': np.float64(-5.07332), '10%': np.float64(-4.82668)}
Estimated Break Point index: 1
Estimated Break Date: 2011-06-30


### Custom Lumsdaine-Papell Double Structural Break Unit Root Test
Since the Lumsdaine-Papell (LP) test is not directly available in standard packages like `statsmodels`, we implement it systematically. The LP test extends the Zivot-Andrews framework by allowing for two structural breaks in either the intercept (Model I), the trend (Model II), or both (Model III).

Below, we implement Model III (breaks in both intercept and trend) which is the most comprehensive formulation:
$$y_t = \mu + \beta t + \theta_1 D1_t(\lambda_1) + \gamma_1 DT1_t(\lambda_1) + \theta_2 D2_t(\lambda_2) + \gamma_2 DT2_t(\lambda_2) + \alpha y_{t-1} + \sum_{i=1}^k c_i \Delta y_{t-i} + e_t$$
where $D1_t$ and $D2_t$ are indicator variables for intercept breaks, and $DT1_t$ and $DT2_t$ are trend break variables.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

def lumsdaine_papell_test(series, max_lags=4, trim=0.15):
    """
    Performs a custom Lumsdaine-Papell (1997) unit root test allowing for
    two structural breaks in both intercept and trend (Model III).
    """
    y = np.asarray(series, dtype=float)
    n = len(y)
    dy = np.diff(y)

    # Optimal lag selection using AIC for the baseline ADF model
    best_lag = 0
    best_aic = np.inf
    for lag in range(1, max_lags + 1):
        # We want to regress dy[lag:] on y[lag:-1] and lagged dy values.
        # The target dy[lag:] has length n - 1 - lag.
        lagged_lists = []
        for i in range(lag):
            # Lagged dy values: dy_{t-1}, dy_{t-2}, ..., dy_{t-lag}
            start = lag - i - 1
            end = len(dy) - i - 1
            lagged_lists.append(dy[start:end])

        lagged_dy = np.column_stack(lagged_lists)
        y_prev = y[lag:-1]
        y_target = dy[lag:]
        X = sm.add_constant(np.column_stack([y_prev, lagged_dy]))
        model = sm.OLS(y_target, X).fit()
        if model.aic < best_aic:
            best_aic = model.aic
            best_lag = lag

    k = best_lag

    # Trimming window to ensure statistical properties near boundaries
    start_idx = int(np.floor(n * trim))
    end_idx = int(np.ceil(n * (1 - trim)))

    min_t_stat = np.inf
    best_breaks = (None, None)

    # Grid search for the two break points (Tb1 and Tb2 where Tb1 < Tb2)
    for tb1 in range(start_idx, end_idx - 1):
        for tb2 in range(tb1 + 2, end_idx):
            t = np.arange(1, n + 1)

            # Dummy variables for breaks
            du1 = np.where(t > tb1, 1.0, 0.0)
            dt1 = np.where(t > tb1, t - tb1, 0.0)
            du2 = np.where(t > tb2, 1.0, 0.0)
            dt2 = np.where(t > tb2, t - tb2, 0.0)

            # Target variable length is n - 1 - k
            y_target = dy[k:]
            y_prev = y[k:-1]
            t_aligned = t[k:-1]
            du1_aligned = du1[k:-1]
            dt1_aligned = dt1[k:-1]
            du2_aligned = du2[k:-1]
            dt2_aligned = dt2[k:-1]

            X_vars = [np.ones_like(y_prev), t_aligned, du1_aligned, dt1_aligned, du2_aligned, dt2_aligned, y_prev]

            # Add lagged differences, ensuring they are aligned to length n - 1 - k
            if k > 0:
                for j in range(k):
                    start = k - j - 1
                    end = len(dy) - j - 1
                    X_vars.append(dy[start:end])

            X = np.column_stack(X_vars)
            res = sm.OLS(y_target, X).fit()

            # Index 6 corresponds to y_{t-1} coefficient (alpha)
            t_alpha = res.tvalues[6]

            if t_alpha < min_t_stat:
                min_t_stat = t_alpha
                best_breaks = (tb1, tb2)

    return min_t_stat, best_breaks, k

# Run the test over all three banks in our automated workflow
for bank in target_banks:
    bank_data = filtered_df[filtered_df['bank_name_clean'] == bank.lower()].sort_values(by=date_col)

    if len(bank_data) > 25:
        series = bank_data[rate_col].dropna().values
        if len(series) > 25:
            print(f"\n{'='*50}\nRunning Lumsdaine-Papell Test for: {bank}\n{'='*50}")
            try:
                t_stat, breaks, lag = lumsdaine_papell_test(series)
                b1, b2 = breaks
                date1 = bank_data.iloc[b1][date_col].strftime('%Y-%m-%d') if b1 is not None else 'N/A'
                date2 = bank_data.iloc[b2][date_col].strftime('%Y-%m-%d') if b2 is not None else 'N/A'

                print(f"LP Test Statistic (t-alpha): {t_stat:.4f}")
                print(f"Optimal Lags Selected: {lag}")
                print(f"Estimated 1st Break Point: Index {b1} ({date1})")
                print(f"Estimated 2nd Break Point: Index {b2} ({date2})")
                print("\nCritical values (Lumsdaine-Papell 1997, Model III):\n  1%: -7.19 | 5%: -6.62 | 10%: -6.33")
                if t_stat < -6.62:
                    print("Result: Reject unit root null hypothesis at the 5% level (Series is Stationary with 2 breaks)")
                else:
                    print("Result: Fail to reject unit root null hypothesis (Series is Non-Stationary)")
            except Exception as e:
                print(f"An error occurred during estimation for {bank}: {e}")
        else:
            print(f"Insufficient data series for {bank}.")
    else:
        print(f"Not enough quarters of data for {bank} to perform a dual-break test safely.")


Running Lumsdaine-Papell Test for: Glacier bank
LP Test Statistic (t-alpha): -6.4680
Optimal Lags Selected: 2
Estimated 1st Break Point: Index 46 (2022-09-30)
Estimated 2nd Break Point: Index 48 (2023-03-31)

Critical values (Lumsdaine-Papell 1997, Model III):
  1%: -7.19 | 5%: -6.62 | 10%: -6.33
Result: Fail to reject unit root null hypothesis (Series is Non-Stationary)

Running Lumsdaine-Papell Test for: Huntington National Bank
LP Test Statistic (t-alpha): -3.6680
Optimal Lags Selected: 1
Estimated 1st Break Point: Index 15 (2014-12-31)
Estimated 2nd Break Point: Index 36 (2020-03-31)

Critical values (Lumsdaine-Papell 1997, Model III):
  1%: -7.19 | 5%: -6.62 | 10%: -6.33
Result: Fail to reject unit root null hypothesis (Series is Non-Stationary)

Running Lumsdaine-Papell Test for: Manufacturers and Traders Trust Company
LP Test Statistic (t-alpha): -6.2239
Optimal Lags Selected: 1
Estimated 1st Break Point: Index 36 (2020-03-31)
Estimated 2nd Break Point: Index 47 (2022-12-31)

C